# 03 — Đánh giá cuối và so sánh bốn mô hình

Chỉ chạy sau notebook 01 và 02. Kiểm tra artifact thuộc cùng một lần chuẩn bị, sau đó mở test để đánh giá cuối. Không tune mô hình từ kết quả này. Bảng, hình và phân tích được lưu ngay trong notebook, không ghi CSV/PNG kết quả ra ngoài.

In [ ]:
from pathlib import Path
import sys
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'src' / 'config.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_DIR = ROOT / 'notebooks' / 'artifacts'
TARGET = 'median_house_value'
SEED = 42

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def verify_files(manifest):
    for name, expected in manifest['files'].items():
        if file_hash(ARTIFACT_DIR / name) != expected:
            raise ValueError(f'Artifact mismatch: {name}. Rerun notebooks in order.')

from src.data.loader import load_housing_data, fetch_dataset_info
from src.data.preprocessing import split_data_8_1_1, fit_preprocessor, transform_features, save_processed_splits
from src.models.persistence import save_preprocessor, load_preprocessor
%matplotlib inline

In [ ]:
from src.models.persistence import load_baseline_models, load_mlp_model
from src.models.mlp_model import predict_mlp
from src.evaluation.metrics import RegressionEvaluator
from src.evaluation.visualization import (plot_actual_vs_predicted, plot_loss_and_mae_curves,
                                         plot_model_comparison_bar, plot_residuals_distribution)
preparation_path = ARTIFACT_DIR / 'preparation_manifest.json'
preparation = json.loads(preparation_path.read_text(encoding='utf-8'))
training = json.loads((ARTIFACT_DIR / 'training_manifest.json').read_text(encoding='utf-8'))
verify_files(preparation)
verify_files(training)
if training['preparation_sha256'] != file_hash(preparation_path):
    raise ValueError('Preparation changed after training. Rerun notebook 02.')
if preparation['raw_sha256'] != file_hash(ROOT / 'data' / 'raw' / 'housing.csv'):
    raise ValueError('Raw dataset changed. Rerun notebooks 01 and 02.')
preprocessor = load_preprocessor(ARTIFACT_DIR / 'preprocessor.joblib')
baselines = load_baseline_models(ARTIFACT_DIR / 'baseline_models.joblib')
model = load_mlp_model(ARTIFACT_DIR / 'mlp_house_price_model.pt')
test_frame, _ = load_housing_data(ARTIFACT_DIR / 'test.csv')
X_test, y_test = test_frame.drop(columns=TARGET), test_frame[TARGET]
test_features = transform_features(preprocessor, X_test)
predictions = {name: estimator.predict(test_features) for name, estimator in baselines.items()}
predictions['MLP PyTorch'] = predict_mlp(model, test_features)
summary = RegressionEvaluator.summarize_models([
    RegressionEvaluator.evaluate_predictions(y_test, values, name)
    for name, values in predictions.items()])
print(f'Đánh giá cuối trên cùng {len(y_test)} mẫu test, cùng preprocessing đã fit train.')
display(summary.style.format({'MSE': '{:,.2f}', 'RMSE': '{:,.2f}', 'MAE': '{:,.2f}', 'R2_Score': '{:.4f}'}))
display(plot_model_comparison_bar(summary))


## Giá thực tế, giá dự đoán và phần dư
Đường chéo là dự đoán lý tưởng. Phần dư bằng actual − predicted; phần dư dương nghĩa là dự đoán thấp hơn giá thật. Histogram chỉ mô tả sai số, không chứng minh phân phối chuẩn.

In [ ]:
for name, values in predictions.items():
    row = summary.loc[summary['Model'] == name].iloc[0]
    display(plot_actual_vs_predicted(y_test, values, name, row.MSE, row.R2_Score))
    display(plot_residuals_distribution(y_test, values, name))

## Các trường hợp hạn chế
Đếm dự đoán âm nhưng không tự clipping. Phân tích riêng giá bị giới hạn ở 500001 USD; sai số vẫn so với nhãn bị capping, không phải giá thật chưa bị giới hạn. Hiển thị số mẫu theo vị trí; nhóm ít mẫu không đủ để khẳng định khả năng tổng quát hóa.

In [ ]:
error_rows = []
for name, values in predictions.items():
    errors = np.abs(y_test.to_numpy() - values)
    capped = y_test.to_numpy() == 500001
    error_rows.append({'Model': name, 'Negative_Predictions': int((values < 0).sum()),
                       'Capped_Count': int(capped.sum()),
                       'Capped_MAE_USD': errors[capped].mean() if capped.any() else np.nan,
                       'Uncapped_MAE_USD': errors[~capped].mean() if (~capped).any() else np.nan})
display(pd.DataFrame(error_rows))
location_errors = X_test[['ocean_proximity']].copy()
location_errors['Absolute_Error_USD'] = np.abs(y_test.to_numpy() - predictions['MLP PyTorch'])
display(location_errors.groupby('ocean_proximity', dropna=False)['Absolute_Error_USD']
        .agg(Count='count', MAE_USD='mean', Median_Error_USD='median'))

## Minh họa ba mẫu test và kiểm tra inference
Đây là các mẫu trong test, không phải ba căn nhà mới thu thập độc lập. Đầu vào thô phải qua đúng preprocessor đã lưu.

In [ ]:
samples = X_test.iloc[:3]
sample_prediction = predict_mlp(model, transform_features(preprocessor, samples))
np.testing.assert_allclose(sample_prediction, predictions['MLP PyTorch'][:3], rtol=1e-5, atol=1e-3)
display(pd.DataFrame({'Actual_USD': y_test.iloc[:3].to_numpy(), 'Predicted_USD': sample_prediction,
                      'Absolute_Error_USD': np.abs(y_test.iloc[:3].to_numpy() - sample_prediction)}))
# Đảo thứ tự cột vẫn phải cho cùng vector và cùng dự đoán.
np.testing.assert_allclose(transform_features(preprocessor, samples.iloc[:, ::-1]),
                           transform_features(preprocessor, samples))
print('Kiểm tra inference từ raw features và thứ tự cột thay đổi: Đạt.')


In [ ]:
best = summary.sort_values('RMSE').iloc[0]
mlp = summary.loc[summary['Model'] == 'MLP PyTorch'].iloc[0]
display(Markdown(f"""### Nhận xét từ kết quả thực nghiệm
- **{best['Model']}** có RMSE thấp nhất trong bốn cấu hình đã thử: **{best['RMSE']:,.2f} USD**.
- MLP: RMSE **{mlp['RMSE']:,.2f} USD**, MAE **{mlp['MAE']:,.2f} USD**, R² **{mlp['R2_Score']:.4f}**.
- Kết quả chỉ áp dụng cho split và cấu hình này; không chứng minh mô hình đứng đầu luôn tốt nhất.
- Target là median khu vực, bị capping; nhóm ISLAND rất ít mẫu. Random split cũng chưa đo được khả năng dự báo ở vùng địa lý hoặc thời điểm mới.
- Target được chuẩn hóa chỉ từ train trong MLP; prediction và metrics đã đổi về USD. Giữ kiến trúc 32–16–1; không điều chỉnh model dựa trên test.
"""))
